## Limitations of sklearn NMF

## 1. Load Data and use NMF
Load the movie ratings data (as in the HW3-recommender-system) and use matrix factorization technique(s) and predict the missing ratings from the test data. Measure the RMSE. You should use sklearn library.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import namedtuple
from sklearn.model_selection import train_test_split
from scipy.sparse import coo_matrix, csr_matrix
from scipy.spatial.distance import jaccard, cosine 
from sklearn.decomposition import NMF
import sklearn.metrics as skm

In [2]:
MV_users = pd.read_csv('data/users.csv')
MV_movies = pd.read_csv('data/movies.csv')
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
Data = namedtuple('Data', ['users','movies','train','test'])
data = Data(MV_users, MV_movies, train, test)

## Use RecSys class developed previously but add NMF

In [19]:
class RecSys():
    def __init__(self,data):
        self.data=data
        self.allusers = list(self.data.users['uID'])
        self.allmovies = list(self.data.movies['mID'])
        self.genres = list(self.data.movies.columns.drop(['mID', 'title', 'year']))
        self.mid2idx = dict(zip(self.data.movies.mID,list(range(len(self.data.movies)))))
        self.uid2idx = dict(zip(self.data.users.uID,list(range(len(self.data.users)))))
        self.Mr=self.rating_matrix()
        self.Mm=None 
        self.sim=np.zeros((len(self.allmovies),len(self.allmovies)))
        
    def rating_matrix(self):
        """
        Convert the rating matrix to numpy array of shape (#allusers,#allmovies)
        """
        ind_movie = [self.mid2idx[x] for x in self.data.train.mID] 
        ind_user = [self.uid2idx[x] for x in self.data.train.uID]
        rating_train = list(self.data.train.rating)
        
        return np.array(coo_matrix((rating_train, (ind_user, ind_movie)), shape=(len(self.allusers), len(self.allmovies))).toarray())


    def predict_everything_to_3(self):
        """
        Predict everything to 3 for the test data
        """
        # Generate an array with 3s against all entries in test dataset
        y_pred = np.ones(self.data.test.rating.shape)*3
        
        return y_pred
        
        
    def predict_to_user_average(self):
        """
        Predict to average rating for the user.
        Returns numpy array of shape (#users,)
        """
        # Generate an array as follows:
        # 1. Calculate all avg user rating as sum of ratings of user across all movies/number of movies whose rating > 0
        # 2. Return the average rating of users in test data
        
        # Caclulate each user's average rating
        rm = self.Mr
        mask = rm > 0
        row_sums = np.sum(rm * mask, axis=1)
        counts = np.sum(mask, axis=1)
        avg_user_rating = row_sums/counts
        
        # For each user in test data, return user's average rating
        yp = np.zeros(self.data.test.rating.shape)
        for i in range(len(yp)):
            user_idx = self.uid2idx[self.data.test.uID[i]]
            yp[i] = avg_user_rating[user_idx]
        
        return yp
    
    def predict_from_sim(self,uid,mid):
        """
        Predict a user rating on a movie given userID and movieID
        """
        # Predict user rating as follows:
        # 1. Get entry of user id in rating matrix
        # 2. Get entry of movie id in sim matrix
        # 3. Employ 1 and 2 to predict user rating of the movie
        idx_user = self.uid2idx[uid]
        ratings_user = self.Mr[idx_user,:]
        idx_movie = self.mid2idx[mid]
        movie_sim = self.sim[idx_movie,:]
        user_rating_biased = np.dot(ratings_user, movie_sim)
        mask_rated = (ratings_user != 0).astype(int)
        weighting_divisor = np.dot(movie_sim, mask_rated)
        user_rating = user_rating_biased/weighting_divisor
        
        return user_rating
    
    def predict(self):
        """
        Predict ratings in the test data. Returns predicted rating in a numpy array of size (# of rows in testdata,)
        """
        yp = np.zeros(self.data.test.rating.shape)
        for i in range(len(self.data.test)):
            yp[i] = self.predict_from_sim(
                self.data.test.uID[i],
                self.data.test.mID[i],
            )
            
        return yp
    
    def rmse(self,yp):
        yp[np.isnan(yp)]=3 #In case there is nan values in prediction, it will impute to 3.
        yt=np.array(self.data.test.rating)
        return np.sqrt(((yt-yp)**2).mean())
    
    def train_nmf(self, k=30, max_iter=300):
        """
        Fit an NMF model and produce a full predicted rating matrix.
        This predicts all missing entries in Mr.
        """
        # Ratings matrix: zeros mean missing
        R = self.Mr.astype(float)

        # Train NMF on ALL observed data
        nmf = NMF(
            n_components=k,
            init="nndsvd",
            max_iter=max_iter,
            random_state=0
        )

        # Factorize R directly
        W = nmf.fit_transform(R)
        H = nmf.components_

        # Full predicted dense matrix
        self.R_pred_nmf = W @ H

        return self.R_pred_nmf
    
    def predict_nmf(self):
        """
        Predict ratings for all rows in self.data.test using the NMF-predicted matrix.
        """
        yp = np.zeros(len(self.data.test))

        for idx in range(len(self.data.test)):
            uid = self.data.test.uID.iloc[idx]
            mid = self.data.test.mID.iloc[idx]
            u = self.uid2idx[uid]
            m = self.mid2idx[mid]
            yp[idx] = self.R_pred_nmf[u, m]

        return yp

## Train NMF and evaluate test data RMSE

In [21]:
rs = RecSys(data)
rs.train_nmf(k=30)
yp = rs.predict_nmf()
print("RMSE on test data: ", rs.rmse(yp))

c:\Code\.venv\lib\site-packages\sklearn\decomposition\_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 300 reached. Increase it to improve convergence.
  warnings.warn(


RMSE on test data:  2.8903612411872954


## 2. Discussion of results

The root-mean-squared-error (RMSE) is ~2.9, which is pretty poor. Since the matrix is so sparse, most of the predictions are near zero. We likely need to do some kind of bias-adjustment similar to how we did in Module 3, since the zeros here don't mean bad ratings, they just mean it wasn't rated. A good way to fix this is to adjust all the ratings by the user average. However in this case, NMF can't process negative values. It may make sense to change all the zero values to the user average, this way they are all positive and the NMF model can process them.